In [1]:
!git clone https://github.com/hoangquan1503/Walmart_sales_forecasting.git

In [2]:
#!pip uninstall -y numpy pandas matplotlib seaborn prophet cmdstanpy torchmetrics scikit-learn lightgbm optuna shap pyarrow joblib streamlit torch && pip install numpy==1.26.4 pandas==2.2.2 matplotlib==3.7.2 seaborn==0.13.2 prophet==1.1.4 cmdstanpy==1.2.5 torchmetrics==1.3.0 scikit-learn==1.5.1 lightgbm==4.3.0 optuna==3.6.1 shap==0.44.1 pyarrow==20.0.0 joblib==1.4.2 streamlit==1.35.0 torch==2.3.0

In [3]:
import matplotlib
print(matplotlib.__version__)

In [4]:
%cd Walmart_sales_forecasting
!git pull

In [5]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter


# Set the path to the file you'd like to load


# Load the latest version
train = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "aslanahmedov/walmart-sales-forecast",
  "train.csv",
)
stores = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "aslanahmedov/walmart-sales-forecast",
  "stores.csv",
)
features = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "aslanahmedov/walmart-sales-forecast",
  "features.csv",
)

df = train.merge(stores, on='Store').merge(features, on=['Store', 'Date'])

df.drop(columns=['IsHoliday_y'], inplace=True)
df.rename(columns={'IsHoliday_x' : 'IsHoliday'}, inplace=True)

print("First 5 records:", df.head())

In [6]:
df['Date'] = pd.to_datetime(df['Date'])
df['Date']

In [7]:
from src.data_cleaning import check_missing_value, fill_missing_promote

check_missing_value(df)



In [8]:
markdown = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
df_filled = fill_missing_promote(df, markdown)



In [9]:
from src.plots import plot_sales

plot_sales(df_filled, 1, 1)

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
summary = (
    df_filled[['Store', 'Dept', 'Weekly_Sales']].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99, 0.999])
    .iloc[1:]
)
plt.figure(figsize=(8, 4)) 
sns.heatmap(summary, cmap='rocket', annot = True)
plt.title("Descriptive Statistics Heatmap")
plt.show()

In [11]:
plt.figure(figsize=(10,4))
sns.histplot(df_filled['Weekly_Sales'], bins=100, kde=True)
plt.title('Histogram of Sales with KDE')
plt.xlabel('Sales')
plt.ylabel('Frequency')
plt.show()

as we can see, there are outliers in this dataset

In [12]:
from src.data_cleaning import corrected_outlier
df_corrected = corrected_outlier(df_filled)
df_corrected

In [13]:
summary = (
    df_corrected[['Store', 'Dept', 'Weekly_Sales']].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99, 0.999])
    .iloc[1:]
)
plt.figure(figsize=(8, 4)) 
sns.heatmap(summary, cmap='rocket', annot = True)
plt.title("Descriptive Statistics Heatmap After Correcting")
plt.show()

In [14]:
store_id = 1
df_store = df_corrected[df_corrected['Store'] == store_id]
dept_list = df_store['Dept'].unique()  


if len(dept_list) > 30:
    dept_list = dept_list[:30]  

n_rows, n_cols = 5, 6
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3), constrained_layout=True)
axes = axes.flatten()

for idx, dept in enumerate(dept_list):
    df_dept = df_store[df_store['Dept'] == dept].sort_values('Date')
    ax = axes[idx]
    ax.plot(df_dept['Date'], df_dept['Weekly_Sales'], color='steelblue', linewidth=1.5)
    ax.set_title(f'Dept {dept}', fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, linestyle='--', alpha=0.3)


for ax in axes[len(dept_list):]:
    ax.axis('off')

fig.suptitle(f'Sales Trends - Store {store_id}', fontsize=16, fontweight='bold', y=1.02)
plt.show()

In [15]:
import os

# position
%cd /content/Walmart_sales_forecasting


os.makedirs("data/processed", exist_ok=True)


save_file = "data/processed/sales_data_preprocessed.csv"

# save file
df_corrected.to_csv(save_file, index=False)

In [16]:
%cd /content/Walmart_sales_forecasting/data/processed/
!ls


In [17]:
import importlib
import src.metrics
importlib.reload(src.metrics)
from src.metrics import weighted_absolute_percentage_error
import inspect
print(inspect.getsource(weighted_absolute_percentage_error))